# STAGE B — MEAD Landmark Packages (per-actor, resumable) · v3.0.0

Mediapipe FaceLandmarker (478 → 68 điểm dlib) + 52 blendshape + head pose, trích từ `MEAD/<actor>/video.tar`.
**Mỗi actor = 1 package độc lập**: `process → validate → package → upload Drive → verify → COMPLETED → actor kế`.

**Cách chạy:** `Runtime ▸ Run all`. Test trước: `MAX_ACTORS=1`, `MAX_CLIPS_PER_ACTOR=5` (chạy trong thư mục `*_test` riêng), hoặc `DRY_RUN=True`.
Sau khi Colab reset: **chạy lại từ cell 1** (biến/hàm không được giữ) — actor xong sẽ SKIP, actor dở sẽ RESUME.

---
## Audit → Problems → Proposed → Implementation

**Current pipeline (v2.2):** 1 vòng lặp chạy ô 12 → `process_actor` → `package_actor` → `upload_actor`; lock + archive + resume phức tạp (~500 dòng).

**Vì sao ô 12 không trích được landmark:**
1. `NameError: ACTORS` — traceback trong file là bản ô 12 *cũ*; runtime mất biến (reset / chạy ô 12 trước ô 1). Ô 12 không đứng một mình được.
2. **Lỗi thật sẽ gặp ngay sau đó:** `_gen_with_priority` duyệt hết tar để sắp xếp, *rồi mới* `extractfile()`. `video.tar` là xz đọc stream (`r|*`) → không seek lùi → mọi clip đọc thất bại. Sửa: **một lượt stream duy nhất**, trích clip ngay khi gặp.

**Các vấn đề khác đã sửa:**

| Vấn đề cũ | Sửa |
|---|---|
| `scan_tar_summary` đọc cả tar xz 2 lần (M003 = 20 GB) | đếm `expected_clips` ngay trong lượt trích |
| `MAX_CLIPS_PER_ACTOR` không áp dụng cho actor mới | áp dụng trong vòng lặp; test mode cô lập thư mục `*_test` |
| Pad/crop về 96 frame, `frame_idx` giả cho frame pad | giữ **toàn bộ frame thật** (trần `MAX_FRAMES=150`), không pad |
| Decode cả clip 1080p vào RAM (~0.5 GB/clip) | resize ngay từng frame, tối đa 150 frame 640×360 |
| `confidence` = 0 khi mediapipe không trả visibility | `NaN` nếu model không cung cấp (không bịa) |
| Xoay landmark trên toạ độ chuẩn hoá 640×360 (bất đẳng hướng) | `landmark_norm` tính trong pixel, **không xoay**; lưu `pose_matrix` để xoay lại sau |
| Template neutral theo (angle, level) — neutral chỉ có 1 mức ⇒ mất delta level 2/3 | template theo angle; delta là dữ liệu **phụ** |
| `ALPHA` nằm trong config hash (đổi ALPHA ⇒ trích lại toàn bộ) | bỏ; derived tính lại từ master |
| `STRICT_PACKAGE` kẹt vĩnh viễn nếu 1 clip hỏng thật | clip hỏng được **ghi vào manifest + errors.json**, actor vẫn hoàn tất |
| JPEG từng frame (vài GB/actor), copy clip 3–4 lần | tắt mặc định; zip trực tiếp (`ZIP_STORED` cho npz) |
| Lock/lease, archive-intent (không cần vì dataset đã chia máy) | bỏ; chỉ giữ đổi tên bản cũ khi config đổi |

**Proposed pipeline:** `NEW → IN_PROGRESS → EXTRACTED → PACKAGED → COMPLETED`.
`COMPLETED` chỉ hợp lệ khi **Drive có `receipt.json`** (ghi sau cùng, sau khi zip trên Drive đã khớp sha256 + mở được + đúng số clip).

```
LOCAL_ROOT/<actor>/clips/*.npz, state.json            (làm việc, mất khi reset)
PERSIST_ROOT/<actor>/ckpt/{state.json, clips/}        (checkpoint mức clip, xoá clip sau khi COMPLETED)
PERSIST_ROOT/<actor>/<actor>_landmark_package.zip     (package độc lập)
PERSIST_ROOT/<actor>/{manifest.json, receipt.json}
```
Trong zip: `manifest.json` (status=PACKAGED — trạng thái lúc đóng gói; xác nhận cuối là `receipt.json`), `actor_metadata.json`, `clips_index.json`, `clips/*.npz`, `statistics/{summary.json,derived.npz}`, `errors/errors.json`.

## 1. Configuration
Sửa `MEAD_ROOT`, `PERSIST_ROOT`, `ACTORS` cho khớp máy này.

In [1]:
# ===========================================================================
# 1. CONFIGURATION  — mọi tham số nằm ở đây
# ===========================================================================
import os, sys, time
IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')

# --- Đường dẫn -------------------------------------------------------------
MEAD_ROOT    = '/content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD'
PERSIST_ROOT = '/content/drive/MyDrive/DAT301m/Data_student/Data_emotion_mead_task_b/stage_b_mead_out'   # Drive (bền)
LOCAL_ROOT   = '/content/stage_b_work'                                                                   # ổ local Colab (nhanh, mất khi reset)
if not IS_COLAB:   # chạy local
    LOCAL_ROOT, PERSIST_ROOT = os.path.abspath('stage_b_work'), os.path.abspath('stage_b_out')

# --- Actor của máy này (dataset đã chia sẵn, không cần lock) ----------------
ACTORS = ["M005", "M012", "M023"]    # [] = tất cả actor có video.tar

ANGLES   = ["front", "left_30", "right_30", "up", "down"]
EMOTIONS = ["neutral", "angry", "disgusted", "fear", "happy", "sad", "surprised", "contempt"]
NEUTRAL  = "neutral"

# --- Trích landmark ----------------------------------------------------------
MAX_FRAMES     = 150      # trần số frame / clip (MEAD ~90 frame => giữ TOÀN BỘ, không crop, không pad)
FRAME_W, FRAME_H = 640, 360
SAVE_FULL_478  = False    # True = lưu thêm 478 điểm raw (float16) -> ~+0.5MB/clip, đủ cho phân tích mesh về sau
SAVE_FRAME_JPG = False    # True = lưu JPEG từng frame (RẤT nặng: vài GB / actor). Mặc định tắt.
JPEG_QUALITY   = 85

# --- TEST MODE ---------------------------------------------------------------
DRY_RUN             = False   # True = chỉ in kế hoạch + xem thử tar, không trích/ghi gì
MAX_ACTORS          = 3     # 0 = không giới hạn
MAX_CLIPS_PER_ACTOR = 0       # >0 = test: chạy trong thư mục *_test riêng (không đụng dữ liệu thật)

# --- Checkpoint / package ------------------------------------------------------
CKPT_EVERY_CLIPS = 25         # sync clip + state lên Drive mỗi N clip
CLIP_RETRIES     = 2          # số lần thử 1 clip trước khi ghi FAILED
UPLOAD_RETRIES   = 3
BUILD_DERIVED    = True       # derived.npz (delta so với neutral) — dữ liệu PHỤ
CLEANUP_AFTER_UPLOAD = True   # xoá local + clip checkpoint trên Drive CHỈ SAU KHI upload đã verify
FORCE_REPROCESS  = []         # ví dụ ['M003'] = làm lại actor đó từ đầu (bản cũ được đổi tên, không xoá)

# --- Phiên bản ------------------------------------------------------------------
PIPELINE_VERSION    = '3.0.0'
FEATURE_VERSION     = 'landmark_v4'
CLIP_SCHEMA_VERSION = 'clip_v4'

TEST_MODE = bool(MAX_CLIPS_PER_ACTOR)
if TEST_MODE:                 # cô lập hoàn toàn dữ liệu test
    LOCAL_ROOT, PERSIST_ROOT = LOCAL_ROOT + '_test', PERSIST_ROOT + '_test'

print('IS_COLAB={} | TEST_MODE={} | DRY_RUN={}'.format(IS_COLAB, TEST_MODE, DRY_RUN))
print('MEAD_ROOT   :', MEAD_ROOT)
print('LOCAL_ROOT  :', LOCAL_ROOT)
print('PERSIST_ROOT:', PERSIST_ROOT)

IS_COLAB=True | TEST_MODE=False | DRY_RUN=False
MEAD_ROOT   : /content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD
LOCAL_ROOT  : /content/stage_b_work
PERSIST_ROOT: /content/drive/MyDrive/DAT301m/Data_student/Data_emotion_mead_task_b/stage_b_mead_out


## 2. Drive / Environment

In [2]:
from google.colab import drive
drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

shortcut_path = '/content/drive/MyDrive/DAT301m/Data_examble/Data_MEAD'

# Kiểm tra xem đường dẫn có phải là một liên kết ảo (symlink) không
if os.path.islink(shortcut_path):
    # Đọc đường dẫn thực sự ẩn phía sau shortcut
    real_path = os.readlink(shortcut_path)
    print("Đường dẫn gốc thực sự của thư mục là:\n", real_path)

    # Liệt kê file trong đường dẫn thực sự
    print("\nDanh sách file bên trong:")
    print(os.listdir(real_path))
else:
    print("Đối tượng này không phải là liên kết ảo.")

Đường dẫn gốc thực sự của thư mục là:
 /content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD

Danh sách file bên trong:
['M013', 'M019', 'M022', 'M007', 'M009', 'M011', 'M012', 'M023', 'M030', 'M027', 'M028', 'M026', 'M029', 'M025', 'M024', 'M037', 'M031', 'M041', 'M033', 'M035', 'M039', 'M042', 'M034', 'M032', 'M040', 'W017', 'W023', 'W021', 'W015', 'W016', 'W014', 'W009', 'W019', 'W018', 'W011', 'W028', 'W025', 'W029', 'W038', 'W033', 'W024', 'W036', 'W037', 'W035', 'W026', 'W040', 'M005', 'M003']


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/DAT301m/Data_student/Data_emotion_mead_task_b/stage_b_mead_out'))

['M003', 'M011', 'M022', 'W009', 'M007', 'W025', 'W026', 'W029', 'M009', 'W014', 'M026', 'M013', 'M030', 'W015', 'W016', 'M034', 'M040', 'W011', 'M019', 'W018', 'M024', 'W017', 'W023', 'M025', 'W019', 'M028', 'W021', 'W024', 'W028', 'W033', 'W035', 'W037', 'W036', 'W038', 'M029', 'W040', 'M031', 'M032', 'M033', 'M035', 'M005', 'M037', 'M039', 'M041', 'M042', 'M012']


In [ ]:
# ===========================================================================
# 2. DRIVE / ENVIRONMENT
# ===========================================================================
import io, gc, json, shutil, zipfile, hashlib, re, tarfile, traceback, subprocess, importlib.util
import numpy as np

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('GLOG_minloglevel', '3')

if importlib.util.find_spec('mediapipe') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe'])

if IS_COLAB and not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

for d in (LOCAL_ROOT, PERSIST_ROOT, os.path.join(LOCAL_ROOT, 'tmp')):
    os.makedirs(d, exist_ok=True)

print('MEAD_ROOT  tồn tại:', os.path.isdir(MEAD_ROOT))
print('PERSIST    ghi được:', os.access(PERSIST_ROOT, os.W_OK))
print('Đĩa trống local: {:.1f} GB'.format(shutil.disk_usage(LOCAL_ROOT).free / 1e9))

MEAD_ROOT  tồn tại: True
PERSIST    ghi được: True
Đĩa trống local: 75.3 GB


## 3. Model loading
Giữ nguyên model cũ, load **một lần** cho cả runtime.

In [ ]:
# ===========================================================================
# 3. MODEL LOADING  (mediapipe FaceLandmarker — giữ nguyên model cũ, load 1 lần / runtime)
# ===========================================================================
MODEL_URL  = ('https://storage.googleapis.com/mediapipe-models/face_landmarker/'
              'face_landmarker/float16/1/face_landmarker.task')
MODEL_PATH = os.path.join(LOCAL_ROOT, 'face_landmarker.task')
_LANDMARKER = None

def get_landmarker():
    global _LANDMARKER
    if _LANDMARKER is not None:
        return _LANDMARKER
    if not (os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 1_000_000):
        import urllib.request
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    from mediapipe.tasks import python as mpp
    from mediapipe.tasks.python import vision
    opt = vision.FaceLandmarkerOptions(
        base_options=mpp.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=vision.RunningMode.IMAGE,       # IMAGE = stateless, an toàn khi resume
        num_faces=1,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True)
    _LANDMARKER = vision.FaceLandmarker.create_from_options(opt)
    return _LANDMARKER

def run_detector(rgb):
    """RGB uint8 (H,W,3) -> kết quả mediapipe. (Tách riêng để test được bằng detector giả.)"""
    import mediapipe as mp
    img = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
    return get_landmarker().detect(img)

if not globals().get('_SKIP_MODEL_INIT'):
    get_landmarker()
    print('FaceLandmarker sẵn sàng: 478 điểm -> 68 điểm dlib + 52 blendshape + ma trận pose 4x4')

FaceLandmarker sẵn sàng: 478 điểm -> 68 điểm dlib + 52 blendshape + ma trận pose 4x4


## 4. Dataset discovery

In [ ]:
# ===========================================================================
# 4. DATASET DISCOVERY  (+ hằng số landmark)
# ===========================================================================
# 478 mediapipe -> 68 điểm kiểu dlib (giữ nguyên mapping cũ)
LMK68 = [127,234,93,132,58,172,136,150,149,176,148,152,377,400,378,379,365,
         70,63,105,66,107, 336,296,334,293,300, 168,6,197,195,5,4,1,19,94,
         33,160,158,133,153,144, 362,385,387,263,373,380,
         61,39,37,0,267,269,291,405,314,17,84,181, 78,82,13,312,308,317,14,87]
assert len(LMK68) == 68
EYE_L, EYE_R = 36, 45
# Nhóm vùng mặt (chỉ số trong bộ 68) — dùng khi phân tích emotion:
GROUPS = {'jaw': list(range(0, 17)), 'brow': list(range(17, 27)), 'nose': list(range(27, 36)),
          'eye': list(range(36, 48)), 'mouth': list(range(48, 68))}

ANGLE_ID = {a: i for i, a in enumerate(ANGLES)}
EMO_ID   = {e: i for i, e in enumerate(EMOTIONS)}

_MEMBER_RE = re.compile(r'(?:^|/)video/([^/]+)/([^/]+)/level_(\d+)/([^/]+)\.mp4$')

def parse_member(name):
    """'video/<angle>/<emotion>/level_<L>/<NNN>.mp4' -> (angle, emotion, level, sentence) hoặc None."""
    m = _MEMBER_RE.search(name)
    return (m.group(1), m.group(2), int(m.group(3)), m.group(4)) if m else None

def make_clip_id(actor, angle, emo, level, sentence):
    return '{}__{}__{}__L{}__{}'.format(actor, angle, emo, level, sentence)

def video_tar(actor):
    return os.path.join(MEAD_ROOT, actor, 'video.tar')

def discover_actors():
    if not os.path.isdir(MEAD_ROOT):
        return []
    return sorted(d for d in os.listdir(MEAD_ROOT) if os.path.isfile(video_tar(d)))

def peek_tar(actor, n=8):
    """Xem n entry đầu của video.tar + kết quả parse (chẩn đoán 'không ra clip nào')."""
    shown = 0
    with tarfile.open(video_tar(actor), 'r|*') as t:
        for m in t:
            if not m.isfile():
                continue
            print('  {:70s} -> {}'.format(m.name[-70:], parse_member(m.name)))
            shown += 1
            if shown >= n:
                break

if not ACTORS:
    ACTORS = discover_actors()
missing = [a for a in ACTORS if not os.path.isfile(video_tar(a))]
print('Actor sẽ xử lý:', len(ACTORS), '| thiếu video.tar:', missing)
for a in ACTORS[:5]:
    if os.path.isfile(video_tar(a)):
        print('  {} video.tar = {:.2f} GB'.format(a, os.path.getsize(video_tar(a)) / 1e9))

Actor sẽ xử lý: 3 | thiếu video.tar: []
  M005 video.tar = 6.58 GB
  M012 video.tar = 6.30 GB
  M023 video.tar = 8.37 GB


## 5. Checkpoint / Manifest

In [ ]:
# ===========================================================================
# 5. CHECKPOINT / MANIFEST
#    state.json (per actor) = nguồn sự thật. Lưu local + sync định kỳ lên Drive.
#    Trạng thái: NEW -> IN_PROGRESS -> EXTRACTED -> PACKAGED -> COMPLETED
#    COMPLETED chỉ có nghĩa khi Drive có receipt.json (ghi SAU KHI verify upload).
# ===========================================================================
def now_iso():  return time.strftime('%Y-%m-%dT%H:%M:%S')
def log(*a):    print(*a, flush=True)

# --- đường dẫn (đọc LOCAL_ROOT/PERSIST_ROOT lúc gọi) -------------------------
def local_dir(a):     return os.path.join(LOCAL_ROOT, a)
def local_clips(a):   return os.path.join(LOCAL_ROOT, a, 'clips')
def local_state(a):   return os.path.join(LOCAL_ROOT, a, 'state.json')
def local_zip(a):     return os.path.join(LOCAL_ROOT, a, '{}_landmark_package.zip'.format(a))
def clip_path(a, cid): return os.path.join(local_clips(a), cid + '.npz')
def drive_dir(a):     return os.path.join(PERSIST_ROOT, a)
def drive_zip(a):     return os.path.join(PERSIST_ROOT, a, '{}_landmark_package.zip'.format(a))
def drive_receipt(a): return os.path.join(PERSIST_ROOT, a, 'receipt.json')
def ckpt_clips(a):    return os.path.join(PERSIST_ROOT, a, 'ckpt', 'clips')
def ckpt_state(a):    return os.path.join(PERSIST_ROOT, a, 'ckpt', 'state.json')

def config_hash():
    """Băm các tham số ảnh hưởng nội dung MASTER (không gồm tham số derived)."""
    keys = ['ANGLES', 'EMOTIONS', 'MAX_FRAMES', 'FRAME_W', 'FRAME_H', 'SAVE_FULL_478',
            'SAVE_FRAME_JPG', 'JPEG_QUALITY', 'FEATURE_VERSION', 'CLIP_SCHEMA_VERSION', 'LMK68']
    blob = json.dumps({k: globals()[k] for k in keys}, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]

# --- IO an toàn ---------------------------------------------------------------
def write_json(path, obj):
    """Ghi atomic: tmp -> fsync -> os.replace (crash giữa chừng không làm hỏng file đích)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=str)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def read_json(path, default=None):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        return default
    except Exception as e:
        log('    [WARN] JSON hỏng {}: {}'.format(path, repr(e)[:80]))
        return default

def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def copy_atomic(src, dst, expect_sha=None):
    """Copy -> dst.part -> kiểm tra (size/sha) -> os.replace. Không bao giờ để dst nửa vời."""
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + '.part'
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != os.path.getsize(src):
        raise IOError('size lệch sau khi copy: ' + dst)
    if expect_sha and sha256_file(tmp) != expect_sha:
        raise IOError('sha256 lệch sau khi copy: ' + dst)
    os.replace(tmp, dst)

# --- state ----------------------------------------------------------------------
def new_state(a):
    return {'actor_id': a, 'status': 'NEW', 'config_hash': config_hash(),
            'pipeline_version': PIPELINE_VERSION, 'feature_version': FEATURE_VERSION,
            'created_at': now_iso(), 'updated_at': now_iso(),
            'expected_clips': None, 'scan_complete': False,
            'clips': {}, 'errors': [], 'blendshape_names': None, 'package': None}

def load_state(a):
    """Lấy bản MỚI HƠN giữa local và Drive checkpoint (sau reset chỉ còn bản Drive)."""
    c = [s for s in (read_json(local_state(a)), read_json(ckpt_state(a))) if s]
    return max(c, key=lambda s: s.get('updated_at', '')) if c else None

def save_state(a, st, drive=False):
    st['updated_at'] = now_iso()
    write_json(local_state(a), st)
    if drive:
        try:
            write_json(ckpt_state(a), st)
        except Exception as e:
            log('    [WARN] chưa ghi được state lên Drive: {}'.format(repr(e)[:80]))
            return False
    return True

def state_stats(st):
    cl = st['clips'].values()
    ok = [c for c in cl if c['status'] == 'PROCESSED']
    return {'processed_clips': len(ok),
            'failed_clips': sum(1 for c in cl if c['status'] == 'FAILED'),
            'total_frames': sum(c['n_frames'] for c in ok),
            'valid_frames': sum(c['n_valid'] for c in ok)}

def sync_checkpoint(a, st):
    """Đẩy clip MỚI + state lên Drive. Lỗi Drive chỉ cảnh báo (local vẫn giữ để thử lại)."""
    try:
        os.makedirs(ckpt_clips(a), exist_ok=True)
        n = 0
        for cid, rec in st['clips'].items():
            if rec['status'] == 'PROCESSED' and not rec.get('synced'):
                copy_atomic(clip_path(a, cid), os.path.join(ckpt_clips(a), cid + '.npz'))
                rec['synced'] = True; n += 1
        ok = save_state(a, st, drive=True)
        if n: log('    [ckpt] +{} clip -> Drive'.format(n))
        return ok
    except Exception as e:
        log('    [WARN] sync checkpoint lỗi (giữ local, thử lại sau): {}'.format(repr(e)[:100]))
        return False

def restore_clips(a, st):
    """Đảm bảo mọi clip PROCESSED có file local (kéo từ Drive nếu runtime vừa reset).
    Clip không khôi phục được bị loại khỏi state để được trích lại. Trả (restored, dropped)."""
    os.makedirs(local_clips(a), exist_ok=True)
    restored = dropped = 0
    for cid, rec in list(st['clips'].items()):
        if rec['status'] != 'PROCESSED' or os.path.isfile(clip_path(a, cid)):
            continue
        src = os.path.join(ckpt_clips(a), cid + '.npz')
        try:
            if os.path.isfile(src):
                copy_atomic(src, clip_path(a, cid)); restored += 1; continue
        except Exception as e:
            log('    [WARN] không khôi phục được {}: {}'.format(cid, repr(e)[:60]))
        del st['clips'][cid]; dropped += 1
    if restored or dropped:
        log('    [resume] khôi phục {} clip từ Drive | {} clip phải trích lại'.format(restored, dropped))
    return restored, dropped

def archive_actor(a):
    """Config đổi / FORCE_REPROCESS: ĐỔI TÊN (không xoá) dữ liệu cũ rồi làm lại từ đầu."""
    tag = '.old_' + time.strftime('%Y%m%d_%H%M%S')
    for d in (drive_dir(a), local_dir(a)):
        if os.path.isdir(d):
            os.replace(d, d + tag)
    log('    [archive] dữ liệu cũ của {} được đổi tên *{} (không xoá)'.format(a, tag))

## 6. Clip processing

## Schema của mỗi clip `.npz` (MASTER)

| Key | Shape | Ý nghĩa |
|---|---|---|
| `landmark_raw` | (N,68,3) | toạ độ mediapipe gốc |
| `landmark_norm` | (N,68,3) | tâm 2 mắt = 0, đơn vị = khoảng cách 2 mắt (tính trong pixel, không xoay) |
| `face_scale` | (N,) | khoảng cách 2 mắt (pixel) |
| `pose`, `pose_matrix` | (N,3), (N,4,4) | pitch/yaw/roll (độ), ma trận 4×4 của model |
| `blendshape` | (N,52) | 52 blendshape (tên trong `actor_metadata.json`) |
| `confidence`, `bbox` | (N,68), (N,4) | confidence (NaN nếu model không cung cấp), bbox pixel |
| `valid_mask`, `frame_idx`, `timestamp` | (N,) | frame detect được / chỉ số frame gốc / giây |
| `meta_json` | scalar str | actor, clip_id, angle, emotion, level, sentence, source, fps… |
| `landmark478_raw` | (N,478,3) f16 | chỉ khi `SAVE_FULL_478=True` |

Frame mất mặt: mọi mảng hình học = `NaN`, `valid_mask=0`. Nhãn lấy từ đường dẫn trong tar, không tự tạo.

In [ ]:
# ===========================================================================
# 6. CLIP PROCESSING  (1 clip -> 1 file npz, giữ TOÀN BỘ frame + raw + normalized + temporal)
#
#  landmark_raw   (N,68,3)  toạ độ mediapipe gốc (x,y chuẩn hoá theo ảnh, z)
#  landmark_norm  (N,68,3)  tịnh tiến về tâm 2 mắt + chia khoảng cách 2 mắt, tính trong PIXEL
#                           (KHÔNG xoay: xoay trên toạ độ chuẩn hoá bất đẳng hướng 640x360 là sai;
#                            pose_matrix được lưu để xoay lại sau nếu cần)
#  face_scale     (N,)      khoảng cách 2 mắt (pixel) — để khôi phục px từ norm
#  pose (N,3) pitch/yaw/roll độ | pose_matrix (N,4,4) | blendshape (N,52)
#  confidence     (N,68)    CHỈ có giá trị nếu mediapipe trả visibility/presence, nếu không = NaN
#  bbox (N,4) px | valid_mask (N,) | frame_idx (N,) | timestamp (N,) giây | meta_json
#  Frame mất mặt: mọi mảng hình học = NaN, valid_mask = 0 (KHÔNG forward-fill ngầm).
# ===========================================================================
N_BLEND = 52

def euler_xyz(R):
    sy = np.sqrt(R[0, 0] ** 2 + R[1, 0] ** 2)
    if sy > 1e-6:
        p, y, r = np.arctan2(-R[2, 0], sy), np.arctan2(R[1, 0], R[0, 0]), np.arctan2(R[2, 1], R[2, 2])
    else:
        p, y, r = np.arctan2(-R[2, 0], sy), 0.0, np.arctan2(-R[1, 2], R[1, 1])
    return np.degrees([p, y, r]).astype(np.float32)

def normalize_landmarks(raw):
    """(68,3) raw -> (norm (68,3), scale). Làm việc trong PIXEL để giữ đúng tỉ lệ x/y/z."""
    p = raw.astype(np.float64).copy()
    p[:, 0] *= FRAME_W; p[:, 1] *= FRAME_H; p[:, 2] *= FRAME_W      # z của mediapipe cùng tỉ lệ với x
    c = 0.5 * (p[EYE_L] + p[EYE_R])
    d = np.linalg.norm(p[EYE_R, :2] - p[EYE_L, :2]) + 1e-6
    return ((p - c) / d).astype(np.float32), float(d)

def decode_frames(data, tag):
    """bytes mp4 -> (frames BGR đã resize (<=MAX_FRAMES), fps, tổng frame nguồn). Không giữ frame 1080p."""
    import cv2
    tmp = os.path.join(LOCAL_ROOT, 'tmp', 'src_{}.mp4'.format(tag))
    with open(tmp, 'wb') as f:
        f.write(data)
    cap = cv2.VideoCapture(tmp)
    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        fps = float(fps) if fps and 1.0 <= fps <= 240.0 else 30.0
        n_src = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        frames = []
        while len(frames) < MAX_FRAMES:
            ok, fr = cap.read()
            if not ok:
                break
            frames.append(cv2.resize(fr, (FRAME_W, FRAME_H), interpolation=cv2.INTER_AREA))
    finally:
        cap.release()
        try: os.remove(tmp)
        except OSError: pass
    return frames, fps, max(n_src, len(frames))

def parse_detection(res):
    """Kết quả mediapipe -> dict mảng cho 1 frame, hoặc None nếu không thấy mặt."""
    if not res.face_landmarks:
        return None
    fl = res.face_landmarks[0]
    xyz = np.array([[p.x, p.y, p.z] for p in fl], np.float32)           # (478,3)
    raw = xyz[LMK68]
    norm, scale = normalize_landmarks(raw)
    vis = [getattr(fl[i], 'visibility', None) for i in LMK68]
    pre = [getattr(fl[i], 'presence', None) for i in LMK68]
    if all(v is not None for v in vis) and all(v is not None for v in pre):
        conf = np.clip(0.5 * (np.array(vis, np.float32) + np.array(pre, np.float32)), 0, 1)
    else:
        conf = np.full(68, np.nan, np.float32)                           # model không cung cấp -> NaN (không bịa)
    px = xyz[:, :2] * np.array([FRAME_W, FRAME_H], np.float32)
    bbox = np.array([px[:, 0].min(), px[:, 1].min(), px[:, 0].max(), px[:, 1].max()], np.float32)
    if res.facial_transformation_matrixes:
        M = np.array(res.facial_transformation_matrixes[0], np.float32).reshape(4, 4)
        R = M[:3, :3] / (np.linalg.norm(M[:3, :3], axis=0, keepdims=True) + 1e-9)   # bỏ scale trước khi lấy góc
        pose = euler_xyz(R)
    else:
        M, pose = np.full((4, 4), np.nan, np.float32), np.full(3, np.nan, np.float32)
    names = None
    if res.face_blendshapes:
        bl = np.array([c.score for c in res.face_blendshapes[0]], np.float32)
        names = [getattr(c, 'category_name', '') for c in res.face_blendshapes[0]]
        bl = np.pad(bl, (0, max(0, N_BLEND - len(bl))))[:N_BLEND]
    else:
        bl = np.full(N_BLEND, np.nan, np.float32)
    return {'raw': raw, 'norm': norm, 'scale': scale, 'pose': pose, 'M': M, 'bl': bl,
            'conf': conf, 'bbox': bbox, 'xyz478': xyz, 'bl_names': names}

def _nan(*shape):
    return np.full(shape, np.nan, np.float32)

def extract_clip(data, tag, meta_in):
    """bytes mp4 + metadata MEAD -> (arrays dict, meta dict). Raise nếu không đọc được video."""
    import cv2
    frames, fps, n_src = decode_frames(data, tag)
    if not frames:
        raise RuntimeError('video không có frame nào')
    N = len(frames)
    L = {k: [] for k in ('raw', 'norm', 'scale', 'pose', 'M', 'bl', 'conf', 'bbox', 'x478', 'valid', 'jpg')}
    bl_names, n_det = None, 0
    for fr in frames:
        if SAVE_FRAME_JPG:
            ok, buf = cv2.imencode('.jpg', fr, [int(cv2.IMWRITE_JPEG_QUALITY), int(JPEG_QUALITY)])
            L['jpg'].append(buf.tobytes() if ok else b'')
        d = parse_detection(run_detector(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
        if d is None:
            L['raw'].append(_nan(68, 3)); L['norm'].append(_nan(68, 3)); L['scale'].append(np.nan)
            L['pose'].append(_nan(3)); L['M'].append(_nan(4, 4)); L['bl'].append(_nan(N_BLEND))
            L['conf'].append(_nan(68)); L['bbox'].append(_nan(4)); L['x478'].append(_nan(478, 3))
            L['valid'].append(0)
        else:
            n_det += 1
            L['raw'].append(d['raw']); L['norm'].append(d['norm']); L['scale'].append(d['scale'])
            L['pose'].append(d['pose']); L['M'].append(d['M']); L['bl'].append(d['bl'])
            L['conf'].append(d['conf']); L['bbox'].append(d['bbox']); L['x478'].append(d['xyz478'])
            L['valid'].append(1)
            bl_names = bl_names or d['bl_names']
    del frames
    meta = dict(meta_in, schema=CLIP_SCHEMA_VERSION, frame_size=[FRAME_W, FRAME_H], src_fps=fps,
                n_src_frames=n_src, n_frames=N, n_detected=n_det, truncated=bool(n_src > N))
    A = {'landmark_raw': np.stack(L['raw']), 'landmark_norm': np.stack(L['norm']),
         'face_scale': np.array(L['scale'], np.float32),
         'pose': np.stack(L['pose']), 'pose_matrix': np.stack(L['M']), 'blendshape': np.stack(L['bl']),
         'confidence': np.stack(L['conf']), 'bbox': np.stack(L['bbox']),
         'valid_mask': np.array(L['valid'], np.uint8),
         'frame_idx': np.arange(N, dtype=np.int32),                      # chỉ số frame trong video gốc
         'timestamp': (np.arange(N) / fps).astype(np.float32),           # giây
         'src_fps': np.array([fps], np.float32),
         'meta_json': np.array(json.dumps(meta, ensure_ascii=False))}
    if SAVE_FULL_478:
        A['landmark478_raw'] = np.stack(L['x478']).astype(np.float16)
    if SAVE_FRAME_JPG:
        off = np.zeros(N + 1, np.int64); off[1:] = np.cumsum([len(b) for b in L['jpg']])
        A['video_jpg'] = np.frombuffer(b''.join(L['jpg']), np.uint8).copy(); A['video_jpg_offsets'] = off
    meta['bl_names'] = bl_names
    return A, meta

def save_clip(a, cid, A):
    """Ghi atomic (tmp -> replace). Trả (path, sha256)."""
    os.makedirs(local_clips(a), exist_ok=True)
    path = clip_path(a, cid)
    tmp = path + '.tmp'
    with open(tmp, 'wb') as f:
        np.savez_compressed(f, **A)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    return path, sha256_file(path)

print('Đã định nghĩa: extract_clip / save_clip (raw + norm + pose + blendshape + bbox + confidence + timestamp)')

Đã định nghĩa: extract_clip / save_clip (raw + norm + pose + blendshape + bbox + confidence + timestamp)


## 7. Actor processing

In [ ]:
# ===========================================================================
# 7. ACTOR PROCESSING  (stream video.tar MỘT LẦN, trích từng clip, checkpoint mức clip)
#    video.tar là xz => chỉ đọc tuần tự. Clip đã xong được bỏ qua ngay ở header (không giải mã).
# ===========================================================================
def process_actor(a, st=None):
    st = st or load_state(a) or new_state(a)
    restore_clips(a, st)
    st['status'] = 'IN_PROGRESS'
    st['config_hash'] = config_hash()
    save_state(a, st, drive=True)

    t0 = time.time()
    n_seen = n_new = n_fail = n_skip = 0
    since_sync = 0
    limit_hit = False
    done = sum(1 for c in st['clips'].values() if c['status'] == 'PROCESSED')
    log('--- process_actor {} | đã có {} clip | giới hạn={}'.format(a, done, MAX_CLIPS_PER_ACTOR or 'không'))
    try:
        with tarfile.open(video_tar(a), 'r|*') as tar:
            for m in tar:
                if not m.isfile():
                    continue
                pm = parse_member(m.name)
                if pm is None:
                    continue
                angle, emo, level, sent = pm
                if angle not in ANGLES or emo not in EMO_ID:
                    continue
                n_seen += 1
                cid = make_clip_id(a, angle, emo, level, sent)
                rec = st['clips'].get(cid)
                if rec and rec['status'] == 'PROCESSED':
                    n_skip += 1
                    continue
                if MAX_CLIPS_PER_ACTOR and done >= MAX_CLIPS_PER_ACTOR:
                    limit_hit = True
                    break

                data = tar.extractfile(m).read()
                meta_in = {'actor': a, 'clip_id': cid, 'angle': angle, 'emotion': emo,
                           'level': level, 'sentence': sent, 'source': m.name}
                last_err = None
                for attempt in range(1, CLIP_RETRIES + 1):
                    try:
                        A, meta = extract_clip(data, '{}_{}'.format(a, n_seen), meta_in)
                        path, digest = save_clip(a, cid, A)
                        st['clips'][cid] = {
                            'clip_id': cid, 'status': 'PROCESSED', 'actor': a, 'angle': angle,
                            'emotion': emo, 'emotion_id': EMO_ID[emo], 'level': level, 'sentence': sent,
                            'n_frames': meta['n_frames'], 'n_src_frames': meta['n_src_frames'],
                            'n_detected': meta['n_detected'], 'n_valid': int(A['valid_mask'].sum()),
                            'src_fps': meta['src_fps'], 'sha256': digest, 'synced': False,
                            'processed_at': now_iso()}
                        if meta.get('bl_names') and not st.get('blendshape_names'):
                            st['blendshape_names'] = meta['bl_names']
                        n_new += 1; done += 1; last_err = None
                        break
                    except Exception as e:                    # KeyboardInterrupt/BaseException vẫn thoát ra ngoài
                        last_err = (repr(e), traceback.format_exc()[-1200:])
                if last_err:
                    n_fail += 1
                    prev = st['clips'].get(cid, {})
                    st['clips'][cid] = {'clip_id': cid, 'status': 'FAILED', 'actor': a, 'angle': angle,
                                        'emotion': emo, 'level': level, 'sentence': sent,
                                        'fail_count': int(prev.get('fail_count', 0)) + 1,
                                        'last_error': last_err[0], 'failed_at': now_iso()}
                    st['errors'].append({'clip_id': cid, 'error': last_err[0],
                                         'traceback': last_err[1], 'at': now_iso()})
                    log('    [CLIP LỖI] {}: {}'.format(cid, last_err[0][:120]))
                del data

                since_sync += 1
                if CKPT_EVERY_CLIPS and since_sync >= CKPT_EVERY_CLIPS:
                    save_state(a, st); sync_checkpoint(a, st); since_sync = 0; gc.collect()
                    el = time.time() - t0
                    log('    ... {} mới | {} skip | {} lỗi | {:.2f} clip/s'.format(n_new, n_skip, n_fail, n_new / max(el, 1e-6)))
    finally:
        save_state(a, st)                 # crash/Ctrl-C: tiến độ local vẫn được ghi

    if not limit_hit:                     # đã duyệt hết tar -> biết chính xác tổng clip
        st['expected_clips'] = n_seen
        st['scan_complete'] = True
    if n_seen == 0:
        log('    [CẢNH BÁO] không thấy clip nào trong tar — kiểm tra bằng peek_tar("{}")'.format(a))
    st['status'] = 'EXTRACTED'
    save_state(a, st); sync_checkpoint(a, st)
    s = state_stats(st)
    log('--- xong {} | mới={} skip={} lỗi={} | processed={} failed={} expected={} | {:.0f}s'.format(
        a, n_new, n_skip, n_fail, s['processed_clips'], s['failed_clips'], st['expected_clips'], time.time() - t0))
    return st

## 8. Validation

In [ ]:
# ===========================================================================
# 8. VALIDATION  (không chỉ os.path.exists: mở file, kiểm key/shape, số frame, checksum)
# ===========================================================================
REQUIRED_KEYS = ['landmark_raw', 'landmark_norm', 'pose', 'pose_matrix', 'blendshape', 'confidence', 'bbox',
                 'valid_mask', 'frame_idx', 'timestamp', 'face_scale', 'meta_json']

def check_clip(path, rec, checksum=True):
    if not os.path.isfile(path):
        return False, 'thiếu file'
    try:
        if checksum and rec.get('sha256') and sha256_file(path) != rec['sha256']:
            return False, 'checksum lệch'
        with np.load(path) as d:
            for k in REQUIRED_KEYS:
                if k not in d.files:
                    return False, 'thiếu key ' + k
            n = d['landmark_norm'].shape[0]
            if d['landmark_norm'].shape[1:] != (68, 3) or d['landmark_raw'].shape != d['landmark_norm'].shape:
                return False, 'sai shape landmark'
            for k in ('pose', 'blendshape', 'valid_mask', 'frame_idx', 'timestamp', 'bbox'):
                if d[k].shape[0] != n:
                    return False, 'lệch số frame ở ' + k
            vm = d['valid_mask'].astype(bool)
            if vm.any() and not np.isfinite(d['landmark_norm'][vm]).all():
                return False, 'NaN/Inf ở frame valid'
            if n != rec['n_frames'] or int(vm.sum()) != rec['n_valid']:
                return False, 'n_frames/n_valid lệch index'
    except Exception as e:
        return False, 'không đọc được: ' + repr(e)[:80]
    return True, 'ok'

def validate_actor(a, st, fix=False, checksum=True):
    """fix=True: clip hỏng bị xoá khỏi state + đĩa để process_actor trích lại."""
    n_ok, bad = 0, []
    for cid, rec in list(st['clips'].items()):
        if rec['status'] != 'PROCESSED':
            continue
        ok, msg = check_clip(clip_path(a, cid), rec, checksum)
        if ok:
            n_ok += 1
        else:
            bad.append((cid, msg))
            if fix:
                del st['clips'][cid]
                try: os.remove(clip_path(a, cid))
                except OSError: pass
    if fix and bad:
        save_state(a, st)
    s = state_stats(st)
    log('[VALIDATE] {} | ok={} hỏng={} failed={} frames={}'.format(a, n_ok, len(bad), s['failed_clips'], s['total_frames']))
    for cid, msg in bad[:5]:
        log('    - {}: {}'.format(cid, msg))
    return {'n_ok': n_ok, 'n_bad': len(bad), 'bad': bad}

## 9. Packaging

In [ ]:
# ===========================================================================
# 9. PACKAGING  — mỗi actor = 1 zip độc lập
#   <actor>/manifest.json  actor_metadata.json  clips_index.json
#          /clips/<clip_id>.npz           <- MASTER (raw+norm+temporal+metadata)
#          /statistics/summary.json  derived.npz   <- dữ liệu PHỤ
#          /errors/errors.json
# ===========================================================================
def build_derived(a, st, out_path):
    """derived.npz (PHỤ): delta = landmark_norm - template neutral cùng angle của CHÍNH actor.
    Lưu nối tiếp (không pad) + clip_offsets => không có frame giả. Master vẫn là clips/*.npz."""
    ids = sorted(c for c, r in st['clips'].items() if r['status'] == 'PROCESSED')
    acc = {}
    for cid in ids:
        r = st['clips'][cid]
        if r['emotion'] != NEUTRAL:
            continue
        with np.load(clip_path(a, cid)) as d:
            ln, vm = d['landmark_norm'], d['valid_mask'].astype(bool)
        if vm.any():
            s, n = acc.get(r['angle'], (0.0, 0))
            acc[r['angle']] = (s + ln[vm].sum(0, dtype=np.float64), n + int(vm.sum()))
    tpl = {k: (s / n).astype(np.float32) for k, (s, n) in acc.items()}
    D, FI, TS, VM, offs, lab = [], [], [], [], [0], []
    for cid in ids:
        r = st['clips'][cid]
        with np.load(clip_path(a, cid)) as d:
            ln, vm, fi, ts = d['landmark_norm'], d['valid_mask'], d['frame_idx'], d['timestamp']
        delta = (ln - tpl[r['angle']]).reshape(len(ln), -1) if r['angle'] in tpl else np.full((len(ln), 204), np.nan, np.float32)
        D.append(delta.astype(np.float16)); FI.append(fi); TS.append(ts); VM.append(vm)
        offs.append(offs[-1] + len(ln)); lab.append((EMO_ID[r['emotion']], r['level'], ANGLE_ID[r['angle']]))
    np.savez_compressed(out_path, delta_landmark=np.concatenate(D), frame_idx=np.concatenate(FI),
                        timestamp=np.concatenate(TS), valid_mask=np.concatenate(VM),
                        clip_offsets=np.array(offs, np.int64), clip_ids=np.array(ids),
                        labels=np.array(lab, np.int16), label_columns=np.array(['emotion_id', 'level', 'angle_id']),
                        emotions=np.array(EMOTIONS), angles=np.array(ANGLES),
                        template_angles=np.array(list(tpl)), templates=np.stack(list(tpl.values())) if tpl else np.zeros((0, 68, 3), np.float32))
    return {'clips': len(ids), 'frames': int(offs[-1]), 'neutral_templates': sorted(tpl)}

def build_manifest(a, st, status):
    s = state_stats(st)
    return {'actor_id': a, 'status': status,
            'total_clips': s['processed_clips'] + s['failed_clips'],
            'processed_clips': s['processed_clips'], 'failed_clips': s['failed_clips'],
            'total_frames': s['total_frames'], 'valid_frames': s['valid_frames'],
            'expected_clips': st.get('expected_clips'), 'scan_complete': st.get('scan_complete'),
            'pipeline_version': PIPELINE_VERSION, 'feature_version': FEATURE_VERSION,
            'clip_schema': CLIP_SCHEMA_VERSION, 'config_hash': st.get('config_hash'),
            'test_mode': TEST_MODE, 'created_at': now_iso()}

def package_actor(a, st):
    stage = os.path.join(local_dir(a), 'stage')
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(os.path.join(stage, 'statistics')); os.makedirs(os.path.join(stage, 'errors'))
    ids = sorted(c for c, r in st['clips'].items() if r['status'] == 'PROCESSED')
    if not ids:
        raise RuntimeError('không có clip PROCESSED để đóng gói')

    der = build_derived(a, st, os.path.join(stage, 'statistics', 'derived.npz')) if BUILD_DERIVED else None

    import collections as C
    recs = [st['clips'][c] for c in ids]
    summary = {'actor_id': a, **state_stats(st),
               'per_emotion': dict(C.Counter(r['emotion'] for r in recs)),
               'per_angle': dict(C.Counter(r['angle'] for r in recs)),
               'per_level': dict(C.Counter(r['level'] for r in recs)),
               'detect_rate': round(sum(r['n_valid'] for r in recs) / max(1, sum(r['n_frames'] for r in recs)), 4),
               'derived': der}
    write_json(os.path.join(stage, 'statistics', 'summary.json'), summary)
    write_json(os.path.join(stage, 'errors', 'errors.json'),
               {'actor_id': a, 'failed_clips': [c for c, r in st['clips'].items() if r['status'] == 'FAILED'],
                'errors': st['errors']})
    write_json(os.path.join(stage, 'actor_metadata.json'), {
        'actor_id': a, 'source_tar': video_tar(a), 'angles': ANGLES, 'emotions': EMOTIONS, 'neutral': NEUTRAL,
        'frame_size': [FRAME_W, FRAME_H], 'max_frames': MAX_FRAMES, 'landmark_model': 'mediapipe FaceLandmarker (478 -> 68 dlib via LMK68)',
        'lmk68_indices': LMK68, 'groups': GROUPS, 'blendshape_names': st.get('blendshape_names'),
        'normalization': 'landmark_norm = (px - eye_center) / eye_distance, tính trong pixel, không xoay',
        'timestamp': 'frame_idx / src_fps (giây)', 'missing_face': 'NaN + valid_mask=0',
        'labels_source': 'đường dẫn trong video.tar (video/<angle>/<emotion>/level_<L>/<sentence>.mp4); không tự tạo nhãn',
        'config_hash': st.get('config_hash'), 'packaged_at': now_iso()})
    write_json(os.path.join(stage, 'clips_index.json'), {'actor_id': a, 'clips': st['clips']})
    write_json(os.path.join(stage, 'manifest.json'), build_manifest(a, st, 'PACKAGED'))

    zp = local_zip(a); tmp = zp + '.tmp'
    with zipfile.ZipFile(tmp, 'w', allowZip64=True) as z:
        for root, _, files in os.walk(stage):
            for fn in files:
                fp = os.path.join(root, fn)
                comp = zipfile.ZIP_STORED if fn.endswith('.npz') else zipfile.ZIP_DEFLATED   # npz đã nén sẵn
                z.write(fp, os.path.join(a, os.path.relpath(fp, stage)), compress_type=comp)
        for cid in ids:
            z.write(clip_path(a, cid), '{}/clips/{}.npz'.format(a, cid), compress_type=zipfile.ZIP_STORED)
    os.replace(tmp, zp)
    shutil.rmtree(stage, ignore_errors=True)
    st['package'] = {'name': os.path.basename(zp), 'n_bytes': os.path.getsize(zp), 'sha256': sha256_file(zp),
                     'n_clips': len(ids), 'created_at': now_iso()}
    st['status'] = 'PACKAGED'
    save_state(a, st, drive=True)
    log('    [PKG] {} | {} clip | {:.1f} MB | sha256={}…'.format(
        st['package']['name'], len(ids), st['package']['n_bytes'] / 1e6, st['package']['sha256'][:12]))
    return st['package']

## 10. Drive backup

In [ ]:
# ===========================================================================
# 10. DRIVE BACKUP  — upload -> verify -> receipt -> COMPLETED -> cleanup
#     receipt.json là DẤU XÁC NHẬN, ghi SAU CÙNG. Lỗi bất kỳ bước nào => không COMPLETED, giữ local.
# ===========================================================================
def verify_remote(a, pkg):
    """Zip trên Drive: tồn tại, đúng size + sha256, mở được, có manifest và đúng số clip."""
    rz = drive_zip(a)
    if not os.path.isfile(rz):
        return False, 'zip chưa có trên Drive'
    if os.path.getsize(rz) != pkg['n_bytes']:
        return False, 'size Drive lệch'
    if sha256_file(rz) != pkg['sha256']:
        return False, 'sha256 Drive lệch'
    return check_zip_contents(a, rz, pkg['n_clips'])

def check_zip_contents(a, path, n_clips):
    try:
        with zipfile.ZipFile(path) as z:
            names = set(z.namelist())
            for m in ('manifest.json', 'clips_index.json', 'actor_metadata.json'):
                if '{}/{}'.format(a, m) not in names:
                    return False, 'thiếu member ' + m
            man = json.loads(z.read('{}/manifest.json'.format(a)))
            n = sum(1 for x in names if x.startswith(a + '/clips/') and x.endswith('.npz'))
            if n != n_clips or man['processed_clips'] != n_clips:
                return False, 'số clip trong zip ({}) lệch manifest/state ({})'.format(n, n_clips)
    except Exception as e:
        return False, 'không đọc được zip: ' + repr(e)[:80]
    return True, 'ok'

def upload_actor(a, st):
    """Trả (ok, msg). Chỉ ok=True khi bản trên Drive đã verify xong."""
    pkg = st.get('package')
    lz = local_zip(a)
    if not pkg or not os.path.isfile(lz):
        return False, 'chưa có zip local'
    try:
        os.makedirs(drive_dir(a), exist_ok=True)
        ok, _ = verify_remote(a, pkg) if os.path.isfile(drive_zip(a)) else (False, '')
        if not ok:                                              # chưa có / hỏng -> upload (retry + backoff)
            last = None
            for i in range(1, UPLOAD_RETRIES + 1):
                try:
                    log('    [upload] {} (lần {}/{})'.format(drive_zip(a), i, UPLOAD_RETRIES))
                    copy_atomic(lz, drive_zip(a), expect_sha=pkg['sha256']); last = None; break
                except Exception as e:
                    last = e; log('    [upload] lỗi: {}'.format(repr(e)[:100])); time.sleep(min(2 ** i, 10))
            if last:
                return False, 'upload thất bại: ' + repr(last)[:100]
            try: os.sync()
            except Exception: pass
            ok, msg = verify_remote(a, pkg)
            if not ok:
                return False, 'verify Drive thất bại: ' + msg
    except Exception as e:
        return False, 'lỗi Drive: ' + repr(e)[:100]

    # ---- Drive đã verify: ghi receipt (commit) -> state COMPLETED -> cleanup ----
    s = state_stats(st)
    receipt = {'actor_id': a, 'status': 'COMPLETED', 'zip': pkg['name'], 'sha256': pkg['sha256'],
               'n_bytes': pkg['n_bytes'], 'n_clips': pkg['n_clips'], 'failed_clips': s['failed_clips'],
               'total_frames': s['total_frames'], 'config_hash': st['config_hash'],
               'pipeline_version': PIPELINE_VERSION, 'feature_version': FEATURE_VERSION,
               'test_mode': TEST_MODE, 'verified': True, 'completed_at': now_iso()}
    try:
        write_json(os.path.join(drive_dir(a), 'manifest.json'), build_manifest(a, st, 'COMPLETED'))
        write_json(drive_receipt(a), receipt)
    except Exception as e:
        return False, 'không ghi được receipt: ' + repr(e)[:100]
    st['status'] = 'COMPLETED'; st['completed_at'] = receipt['completed_at']
    save_state(a, st, drive=True)
    log('    [COMPLETED] {} đã upload + verify trên Drive'.format(a))
    if s['failed_clips']:
        log('    [LƯU Ý] {} clip FAILED được ghi trong manifest/errors (không bị bỏ qua âm thầm)'.format(s['failed_clips']))
    if CLEANUP_AFTER_UPLOAD:
        shutil.rmtree(os.path.join(drive_dir(a), 'ckpt', 'clips'), ignore_errors=True)   # zip trên Drive đã chứa đủ
        shutil.rmtree(local_dir(a), ignore_errors=True)
        log('    [cleanup] đã xoá file tạm local + clip checkpoint trên Drive')
    return True, 'COMPLETED'

## 11. Resume

In [ ]:
# ===========================================================================
# 11. RESUME  — quyết định SKIP / RESUME / REPACKAGE / UPLOAD / PROCESS bằng BẰNG CHỨNG (không chỉ exists)
# ===========================================================================
def receipt_valid(a, full_sha=False):
    """receipt.json + zip trên Drive còn nguyên (size, mở được, đúng số clip; full_sha=True kiểm cả sha256)."""
    rc = read_json(drive_receipt(a))
    if not rc or rc.get('status') != 'COMPLETED' or not rc.get('verified'):
        return False, None, 'chưa có receipt'
    rz = drive_zip(a)
    if not os.path.isfile(rz):
        return False, rc, 'receipt có nhưng zip mất'
    if os.path.getsize(rz) != rc['n_bytes']:
        return False, rc, 'size zip khác receipt'
    if full_sha and sha256_file(rz) != rc['sha256']:
        return False, rc, 'sha256 zip khác receipt'
    ok, msg = check_zip_contents(a, rz, rc['n_clips'])
    return ok, rc, msg

def decide(a):
    """-> (decision, reason). decision ∈ SKIP | UPLOAD | REPACKAGE | RESUME | PROCESS | REPROCESS"""
    ok, rc, why = receipt_valid(a)
    forced = a in FORCE_REPROCESS
    if ok and not forced:
        if rc.get('config_hash') == config_hash():
            return 'SKIP', 'package hợp lệ trên Drive ({} clip)'.format(rc['n_clips'])
        return 'REPROCESS', 'config đổi so với package cũ'
    if forced:
        return 'REPROCESS', 'FORCE_REPROCESS'
    st = load_state(a)
    if st is None:
        return 'PROCESS', 'chưa có gì'
    if st['config_hash'] != config_hash():
        return 'REPROCESS', 'config đổi so với checkpoint'
    if st['status'] == 'COMPLETED':                      # state nói xong nhưng Drive không còn bằng chứng
        return 'PROCESS', 'state COMPLETED nhưng Drive không verify được ({}) -> làm lại'.format(why)
    if st['status'] == 'PACKAGED' and os.path.isfile(local_zip(a)):
        return 'UPLOAD', 'có zip local, chưa upload/verify xong'
    if st['status'] in ('PACKAGED', 'EXTRACTED'):
        return 'REPACKAGE', 'đã trích xong, chỉ cần đóng gói + upload (không đọc lại tar)'
    n = state_stats(st)['processed_clips']
    return 'RESUME', 'đang dở: đã có {} clip, trích tiếp phần còn lại'.format(n)

def plan_run(actors):
    log('{:<8} {:<10} {}'.format('actor', 'decision', 'reason'))
    for a in actors:
        d, why = decide(a)
        log('{:<8} {:<10} {}'.format(a, d, why))

## 12. Run
Chạy tuần tự từng actor. Nếu báo thiếu hàm/biến: Colab vừa reset → chạy lại từ cell 1.

In [ ]:
# ===========================================================================
# 12. RUN  — Actor xong -> validate -> package -> upload -> verify -> COMPLETED -> actor kế
# ===========================================================================
_need = [n for n in ('ACTORS', 'run_detector', 'process_actor', 'validate_actor', 'package_actor',
                     'upload_actor', 'decide') if n not in globals()]
if _need:
    raise RuntimeError('Thiếu {} — Colab vừa reset runtime. Chạy lại các cell phía trên (Runtime ▸ Run all).'.format(_need))

def run_actor(a):
    res = {'actor': a, 'status': None, 'decision': None, 'msg': '', 'clips': 0, 'failed': 0}
    try:
        d, why = decide(a); res['decision'] = d
        log('\n>>> {} : {} — {}'.format(a, d, why))
        if d == 'SKIP':
            rc = read_json(drive_receipt(a)) or {}
            res.update(status='COMPLETED', clips=rc.get('n_clips', 0), failed=rc.get('failed_clips', 0), msg='đã có package hợp lệ')
            return res
        if DRY_RUN:
            res.update(status='DRY_RUN'); return res

        if d == 'REPROCESS':
            archive_actor(a); d = 'PROCESS'
        must_pack = d != 'UPLOAD'                                # UPLOAD: tái dùng zip local nếu clip không đổi
        if d in ('PROCESS', 'RESUME'):
            st = process_actor(a)
        else:                                                    # REPACKAGE / UPLOAD: không đọc lại tar
            st = load_state(a)
            _, dropped = restore_clips(a, st)
            if dropped:                                          # thiếu clip -> phải trích lại phần thiếu
                save_state(a, st); st = process_actor(a, st); must_pack = True

        for attempt in range(3):                                 # validate; clip hỏng -> trích lại (tối đa 2 lần)
            rep = validate_actor(a, st, fix=True)
            if not rep['n_bad']:
                break
            if attempt == 2:
                raise RuntimeError('vẫn còn clip hỏng sau 2 lần trích lại: {}'.format(rep['bad'][:3]))
            st = process_actor(a, st); must_pack = True
        if rep['n_ok'] == 0:
            raise RuntimeError('không có clip hợp lệ nào')

        if must_pack or not (st.get('package') and os.path.isfile(local_zip(a))):
            package_actor(a, st)
        ok, msg = upload_actor(a, st)
        s = state_stats(st)
        res.update(status='COMPLETED' if ok else 'UPLOAD_FAILED', msg=msg, clips=s['processed_clips'], failed=s['failed_clips'])
        if not ok:
            log('    [!] {} CHƯA COMPLETED: {} — local được giữ, chạy lại để retry'.format(a, msg))
    except Exception as e:
        res.update(status='ERROR', msg=repr(e)[:200])
        log('[LỖI] {}: {}'.format(a, repr(e)))
        traceback.print_exc()
    return res

run_list = list(ACTORS)[:int(MAX_ACTORS)] if MAX_ACTORS else list(ACTORS)

if DRY_RUN:
    log('*** DRY_RUN: chỉ in kế hoạch, KHÔNG trích/ghi/upload ***')
    plan_run(run_list)
    if run_list and os.path.isfile(video_tar(run_list[0])):
        log('\nXem thử tar của {}:'.format(run_list[0])); peek_tar(run_list[0])
    results = []
else:
    results = []
    t_all = time.time()
    for i, a in enumerate(run_list, 1):
        log('\n##### [{}/{}] {} #####'.format(i, len(run_list), a))
        results.append(run_actor(a))
        el = time.time() - t_all
        log('--- {} : {} | tổng {:.0f}s | ETA ~{:.0f}s'.format(a, results[-1]['status'], el, el / i * (len(run_list) - i)))
        gc.collect()
    log('\n' + '=' * 70 + '\nTỔNG KẾT')
    for r in results:
        log('{:<8} {:<14} {:<10} clips={:<6} failed={:<4} {}'.format(r['actor'], r['status'], r['decision'], r['clips'], r['failed'], r['msg'][:60]))
    log('COMPLETED {}/{}'.format(sum(r['status'] == 'COMPLETED' for r in results), len(results)))

## 13. Statistics / basic visualization
Đọc dữ liệu trực tiếp từ package trên Drive. Ví dụ: `for rec, arr in iter_package_clips("M003", emotion="happy"): ...` và `plot_actor_overview("M003")`.

In [ ]:
# ===========================================================================
# 13. STATISTICS / BASIC VISUALIZATION  + đọc dữ liệu TỪ PACKAGE (không cần detector)
# ===========================================================================
def status_table(actors=None):
    actors = actors or ACTORS
    log('{:<8} {:<12} {:>8} {:>7} {:>10} {:>9}'.format('actor', 'status', 'clips', 'failed', 'frames', 'zip MB'))
    n_done = 0
    for a in actors:
        ok, rc, why = receipt_valid(a)
        if ok:
            n_done += 1
            log('{:<8} {:<12} {:>8} {:>7} {:>10} {:>9.0f}'.format(a, 'COMPLETED', rc['n_clips'], rc['failed_clips'], rc['total_frames'], rc['n_bytes'] / 1e6))
        else:
            st = load_state(a)
            if st:
                s = state_stats(st)
                log('{:<8} {:<12} {:>8} {:>7} {:>10} {:>9}'.format(a, st['status'], s['processed_clips'], s['failed_clips'], s['total_frames'], '-'))
            else:
                log('{:<8} {:<12}'.format(a, 'NEW'))
    log('COMPLETED: {}/{}'.format(n_done, len(actors)))

def package_index(a):
    with zipfile.ZipFile(drive_zip(a)) as z:
        return json.loads(z.read('{}/clips_index.json'.format(a)))['clips']

def iter_package_clips(a, emotion=None, level=None, angle=None, sentence=None, keys=None):
    """Đọc clip trực tiếp từ zip trên Drive, lọc theo metadata MEAD. KHÔNG đọc video, KHÔNG chạy detector.
       for rec, arr in iter_package_clips('M003', emotion='happy', level=3): ..."""
    idx = package_index(a)
    with zipfile.ZipFile(drive_zip(a)) as z:
        for cid, r in sorted(idx.items()):
            if r['status'] != 'PROCESSED': continue
            if emotion and r['emotion'] != emotion: continue
            if level is not None and r['level'] != level: continue
            if angle and r['angle'] != angle: continue
            if sentence and r['sentence'] != sentence: continue
            with np.load(io.BytesIO(z.read('{}/clips/{}.npz'.format(a, cid)))) as d:
                yield r, {k: d[k] for k in (keys or d.files)}

def clip_dynamics(arr, key='landmark_norm'):
    """Velocity / acceleration theo timestamp THẬT, chỉ giữa các frame kề nhau đều valid; không nội suy qua gap."""
    x = arr[key].reshape(len(arr[key]), -1).astype(np.float32)
    t = arr['timestamp'].astype(np.float32); v = arr['valid_mask'].astype(bool)
    vel = np.full_like(x, np.nan); acc = np.full_like(x, np.nan)
    dt = np.diff(t)
    ok = v[1:] & v[:-1] & (dt > 0)
    vel[1:][ok] = np.diff(x, axis=0)[ok] / dt[ok, None]
    ok2 = ok[1:] & ok[:-1]
    acc[2:][ok2] = np.diff(vel, axis=0)[1:][ok2] / dt[1:][ok2, None]
    return vel, acc

def plot_actor_overview(a, emotion_list=('neutral', 'happy', 'sad', 'angry'), angle='front', level=None):
    """Thống kê nhãn + quỹ đạo độ mở miệng (blendshape jawOpen nếu có) theo emotion."""
    import matplotlib.pyplot as plt, collections as C
    idx = package_index(a)
    ok = [r for r in idx.values() if r['status'] == 'PROCESSED']
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    c = C.Counter(r['emotion'] for r in ok); ax[0].bar(c.keys(), c.values()); ax[0].set_title('{}: clip / emotion'.format(a)); ax[0].tick_params(axis='x', rotation=45)
    ax[1].hist([r['n_valid'] / max(1, r['n_frames']) for r in ok], bins=20); ax[1].set_title('tỉ lệ frame detect được')
    names = (json.loads(zipfile.ZipFile(drive_zip(a)).read('{}/actor_metadata.json'.format(a))).get('blendshape_names') or [])
    j = names.index('jawOpen') if 'jawOpen' in names else None
    for emo in emotion_list:
        for r, arr in iter_package_clips(a, emotion=emo, angle=angle, level=level, keys=['blendshape', 'timestamp']):
            y = arr['blendshape'][:, j] if j is not None else np.nanmean(np.abs(arr['blendshape']), 1)
            ax[2].plot(arr['timestamp'], y, alpha=.5, label=emo); break               # 1 clip / emotion
    h, l = ax[2].get_legend_handles_labels(); ax[2].legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys())
    ax[2].set_title('jawOpen theo thời gian ({})'.format(angle)); ax[2].set_xlabel('giây')
    plt.tight_layout(); plt.show()

if not DRY_RUN:
    status_table()

## 14. Chứng minh 4 tình huống
Đặt `RUN_DEMO = True` trong cell bên dưới. Demo dùng video + detector **giả** trong thư mục mô phỏng (không đụng dữ liệu thật) nhưng chạy đúng các hàm pipeline thật.
1. **M001** completed + uploaded → reset runtime → `SKIP`, detector không được gọi.
2. **M002** chết giữa chừng → reset → `RESUME` từ checkpoint Drive, chỉ trích phần thiếu.
3. **M003** upload lỗi → không có receipt/COMPLETED, giữ local → retry thành công.
4. **M001** → lọc `emotion=="happy"` + velocity/acceleration **từ package**, không chạy lại detector.

In [ ]:
# ===========================================================================
# 14. CHỨNG MINH 4 TÌNH HUỐNG  (dữ liệu GIẢ + detector GIẢ, thư mục mô phỏng riêng — không đụng dữ liệu thật)
#     Chạy các hàm pipeline THẬT: process_actor / validate / package_actor / upload_actor / decide.
#     Bật bằng RUN_DEMO = True rồi chạy cell này.
# ===========================================================================
RUN_DEMO = False

def _fake_tar(root, a, n_clips_spec):
    import cv2
    os.makedirs(os.path.join(root, a), exist_ok=True)
    with tarfile.open(os.path.join(root, a, 'video.tar'), 'w:xz') as t:
        for (angle, emo, lev, sent) in n_clips_spec:
            p = os.path.join(root, 'tmp_{}.mp4'.format(sent))
            w = cv2.VideoWriter(p, cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (160, 90))
            for i in range(12):
                w.write(np.full((90, 160, 3), (i * 15) % 255, np.uint8))
            w.release()
            t.add(p, arcname='video/{}/{}/level_{}/{}.mp4'.format(angle, emo, lev, sent)); os.remove(p)

class _FakeDetector:
    """Thay run_detector: trả landmark giả có cấu trúc giống mediapipe; đếm số lần gọi, có thể 'chết' tại lần thứ N."""
    def __init__(self): self.calls, self.crash_at = 0, None
    def __call__(self, rgb):
        from types import SimpleNamespace as NS
        self.calls += 1
        if self.crash_at and self.calls == self.crash_at:
            raise KeyboardInterrupt('mô phỏng runtime chết')
        r = np.random.RandomState(self.calls)
        pts = [NS(x=.3 + .4 * r.rand(), y=.3 + .4 * r.rand(), z=.02 * r.randn(), visibility=None, presence=None) for _ in range(478)]
        pts[LMK68[EYE_L]].x, pts[LMK68[EYE_R]].x = .4, .6          # 2 mắt tách nhau -> scale > 0
        return NS(face_landmarks=[pts], facial_transformation_matrixes=[np.eye(4)],
                  face_blendshapes=[[NS(score=float(r.rand()), category_name='bs%d' % i) for i in range(52)]])

def run_demo_cases():
    global MEAD_ROOT, LOCAL_ROOT, PERSIST_ROOT, CKPT_EVERY_CLIPS, DRY_RUN, MAX_CLIPS_PER_ACTOR, run_detector, copy_atomic
    saved = (MEAD_ROOT, LOCAL_ROOT, PERSIST_ROOT, CKPT_EVERY_CLIPS, DRY_RUN, MAX_CLIPS_PER_ACTOR, run_detector, copy_atomic)
    sim = os.path.join(LOCAL_ROOT, '_sim'); shutil.rmtree(sim, ignore_errors=True)
    MEAD_ROOT, LOCAL_ROOT, PERSIST_ROOT = [os.path.join(sim, x) for x in ('mead', 'local', 'drive')]
    for d in (MEAD_ROOT, LOCAL_ROOT, PERSIST_ROOT, os.path.join(LOCAL_ROOT, 'tmp')): os.makedirs(d, exist_ok=True)
    CKPT_EVERY_CLIPS, DRY_RUN, MAX_CLIPS_PER_ACTOR = 2, False, 0
    spec = [('front', 'neutral', 1, '001'), ('front', 'neutral', 1, '002'), ('front', 'happy', 1, '001'),
            ('front', 'happy', 2, '001'), ('front', 'happy', 3, '001'), ('front', 'sad', 1, '001')]
    for a in ('M001', 'M002', 'M003'): _fake_tar(MEAD_ROOT, a, spec)
    det = _FakeDetector(); run_detector = det
    reset = lambda: (shutil.rmtree(LOCAL_ROOT, ignore_errors=True), os.makedirs(os.path.join(LOCAL_ROOT, 'tmp')))
    try:
        log('=' * 70 + '\nCASE 1 — M001 completed + uploaded -> runtime reset -> SKIP\n' + '=' * 70)
        r = run_actor('M001'); assert r['status'] == 'COMPLETED', r
        calls_after_m001 = det.calls
        reset()                                                   # runtime reset: mất toàn bộ local
        d, why = decide('M001'); assert d == 'SKIP', (d, why)
        r = run_actor('M001'); assert det.calls == calls_after_m001, 'SKIP không được chạy detector'
        log('  => PASS: M001 -> {} ({}) | detector không được gọi thêm'.format(d, why))

        log('\n' + '=' * 70 + '\nCASE 2 — M002 xử lý dở -> runtime chết -> reset -> RESUME\n' + '=' * 70)
        det.crash_at = det.calls + 12 * 4 + 5                    # chết giữa clip thứ 4
        try: run_actor('M002')
        except BaseException as e: log('  (mô phỏng) runtime chết:', repr(e))
        det.crash_at = None
        n_synced = sum(1 for c in read_json(ckpt_state('M002'))['clips'].values() if c.get('synced'))
        reset()                                                   # mất local; chỉ còn checkpoint trên Drive
        d, why = decide('M002'); assert d == 'RESUME', (d, why)
        c0 = det.calls
        r = run_actor('M002'); assert r['status'] == 'COMPLETED', r
        redo = (det.calls - c0) // 12
        assert 0 < n_synced < len(spec) and redo == len(spec) - n_synced, (n_synced, redo)
        log('  => PASS: RESUME | checkpoint Drive giữ {} clip, chỉ trích lại {}/{} clip'.format(n_synced, redo, len(spec)))

        log('\n' + '=' * 70 + '\nCASE 3 — M003 upload Drive THẤT BẠI -> không COMPLETED, giữ local, retry\n' + '=' * 70)
        real_copy = copy_atomic
        def broken_drive(src, dst, expect_sha=None):
            if dst.endswith('.zip'): raise OSError('Drive disconnected (mô phỏng)')
            return real_copy(src, dst, expect_sha)
        copy_atomic = broken_drive
        r = run_actor('M003')
        copy_atomic = real_copy
        assert r['status'] == 'UPLOAD_FAILED', r
        assert not os.path.exists(drive_receipt('M003')), 'KHÔNG được có receipt'
        assert load_state('M003')['status'] != 'COMPLETED' and os.path.isfile(local_zip('M003')) and os.path.isdir(local_clips('M003'))
        d, _ = decide('M003'); assert d == 'UPLOAD', d
        log('  upload lỗi -> status={} | local zip + clips còn | decide={}'.format(load_state('M003')['status'], d))
        r = run_actor('M003'); assert r['status'] == 'COMPLETED' and os.path.exists(drive_receipt('M003')), r
        log('  => PASS: retry thành công, chỉ lúc này mới có receipt/COMPLETED')

        log('\n' + '=' * 70 + '\nCASE 4 — dùng package M001 phân tích emotion/dynamics, KHÔNG chạy lại detector\n' + '=' * 70)
        c0 = det.calls
        rows = []
        for rec, arr in iter_package_clips('M001', emotion='happy'):
            vel, acc = clip_dynamics(arr)
            rows.append((rec['clip_id'], rec['level'], arr['landmark_norm'].shape, float(np.nanmean(np.abs(vel))), float(np.nanmean(arr['blendshape'][:, 0]))))
        for r_ in rows: log('  {} | L{} | {} | mean|vel|={:.3f} | blend0={:.3f}'.format(*r_))
        assert rows and det.calls == c0
        log('  => PASS: lọc emotion=="happy" ({} clip) + velocity/acceleration từ package, detector gọi thêm: {}'.format(len(rows), det.calls - c0))
        log('\n4/4 CASE PASS')
    finally:
        (MEAD_ROOT, LOCAL_ROOT, PERSIST_ROOT, CKPT_EVERY_CLIPS, DRY_RUN, MAX_CLIPS_PER_ACTOR, run_detector, copy_atomic) = saved
        shutil.rmtree(sim, ignore_errors=True)

if RUN_DEMO:
    run_demo_cases()
else:
    print('Demo đã định nghĩa. Đặt RUN_DEMO = True ở đầu cell này rồi chạy để chứng minh 4 tình huống.')